# Fock-G1: First-Order Ablation of Aniso-Gaussian Fock-PARFLM — TinyStories

## What this is

The **Fock-G1** first-order ablation defined in §6 of
[`Implicit_vs_Explicit_Damping_and_the_First_vs_Second_Order_Dynamics_Hypothesis.md`](../../../companion_notes/Implicit_vs_Explicit_Damping_and_the_First_vs_Second_Order_Dynamics_Hypothesis.md).
It is the anisotropic-Gaussian analogue of the SPLM-1 ablation
(`first_order_ablation/model_first_order.py`,
`companion_notes/SPLM-1_ablation_pre-registered_protocol.md`).

It answers the **training-phase** question: does the inertial (velocity)
term contribute genuine value *during training*, at matched compute — or does
a well-chosen but fixed step size fully recover the second-order winner?

## The single architectural change

The second-order Fock v2.1 layer update is a damped velocity-Verlet step
(`model_parf_multixi.py::_layer_step`):

    delta = h - h_prev
    denom = 1 + dt * gamma
    h_new = LN( h + delta/denom + dt^2/(m_b*denom) * f )     # second order

Fock-G1 forces the velocity memory `delta = h - h_prev` to **zero** at every
layer, collapsing the update to a pure first-order gradient step:

    h_new = LN( h + beta * f ),   beta = dt^2 / (m_b * (1 + dt * gamma*))

Because gamma is fixed at gamma* (`FIXED_GAMMA`), beta is a **fixed constant**
equal to the second-order anchor's own initial effective step size — so the
ablation isolates the inertial term and nothing else. There is no velocity
buffer and no dynamical role for gamma.

**Everything else is inherited unchanged** from the second-order arm: the
anisotropic Gaussian V_theta (diag + low-rank precision), the pairwise V_phi
(structural_competitive), the multi-channel xi context, the Fock register
creation/destruction gates, the reverse channel, the Fock coupling
regularisation, `force_clamp_max`, the optimiser, the LR schedule, the
grad-clip groups, and every other hyperparameter. Per the SPLM-1 design
discipline, **no independent LR sweep is performed** — Fock-G1 runs at the
same nominal LR as the second-order anchor.

## Implementation note (fidelity to the anchor)

The protocol writes beta with the mean semantic mass `mbar`. This notebook
uses the model's actual **per-token** logfreq mass `m_b` — the same mass the
second-order anchor uses — so beta matches the anchor's per-token initial step
size exactly rather than an averaged scalar. This is strictly more faithful
than the scalar-`mbar` simplification.

## Why gamma* still appears (and how this differs from SPLM-1)

`gamma*` is **not** a damping coefficient here. Damping, by definition,
multiplies a velocity term (`m*h'' = -grad_V - m*gamma*h'`); Fock-G1 zeroes the
velocity memory (`delta := 0`), so there is no velocity for gamma to act on.
What remains is a pure first-order gradient flow `h_new = LN(h + beta*f)`, and
`gamma*` survives **only** as a frozen constant inside the step-size `beta`.

This is a deliberate convention difference from the TinyShakespeare SPLM-1
ablation (`model_first_order.py`), which sets `gamma = 0` entirely and steps
with `h_new = LN(h + dt*f/m)` — gamma absent from the update. Fock-G1 instead
pins `beta = dt^2/(m_b*(1+dt*gamma*))` so its per-layer step equals the
second-order anchor's **layer-0 effective step** exactly. This matched-step
design (§6.3 of the companion note) neutralises the §3.3 confound — that a
higher sweep gamma merely shrinks the effective step — which SPLM-1 controlled
only through the shared nominal LR, not through matched step size. So both
models are genuinely first-order; they differ only in how the single
step-size knob is pinned, and Fock-G1's is the tighter control for this
comparison.

## Comparison anchor and decision rule (TinyStories, d=256)

The second-order anchor is the aniso-Gaussian + fock-reg TinyStories run at
**gamma* = 0.30** (§11.6 of
[`Determining_optimal_gamma_for_Fock-PARFLM.md`](../../../companion_notes/Determining_optimal_gamma_for_Fock-PARFLM.md)),
whose full 20K-step run reached **best val PPL ≈ 9.04**.

| Hypothesis | Operational form | Reading |
|---|---|---|
| H1 (training-time value) | mean Δ ≥ Δmin | inertia adds genuine value beyond any first-order reduction at matched compute |
| H0 (artefact) | \|mean Δ\| < Δmin | the interior gamma* is an effective-step-size artefact; Fock-G1 matches |
| H-1 (refutation) | mean Δ ≤ −Δmin | Fock-G1 outperforms; training-time-value claim falsified for this family |

with Δ = PPL(Fock-G1) − PPL(second-order anchor), averaged over seeds.
**Proposed Δmin = 1.0 PPL** for TinyStories d=256 — comparable to the
adjacent-gamma gap in the §11.6 full runs (γ=0.30 → 9.04 vs γ=0.15 → 10.29, a
1.25 PPL step). The full protocol runs **3 seeds** (SEED = 0, 1, 2) for both
arms; run this notebook once per seed.

## Companion documents

- `companion_notes/Implicit_vs_Explicit_Damping_and_the_First_vs_Second_Order_Dynamics_Hypothesis.md` (§6)
- `companion_notes/SPLM-1_ablation_pre-registered_protocol.md`
- `companion_notes/Determining_optimal_gamma_for_Fock-PARFLM.md` (§11)


In [ ]:
# ── Cell 0: Configuration ──────────────────────────────────────────
import math as _math

SEED = 0    # run once per seed; full protocol uses SEED = 0, 1, 2

# ── First-order ablation (Fock-G1) ──
FIRST_ORDER    = True     # zero velocity memory (delta := 0) => pure gradient step

# ── Architecture ──
D              = 256
L              = 8
VOCAB_SIZE     = 50257
MAX_LEN        = 1024
DT             = 1.0
FIXED_GAMMA    = 0.30     # gamma*: defines beta = dt^2/(m_b*(1+dt*gamma*))
GAMMA_STAR     = FIXED_GAMMA   # alias; gamma has no dynamical role in first order

# ── Second-order comparison anchor (companion Determining_optimal_gamma §11.6) ──
SECOND_ORDER_ANCHOR_PPL = 9.04   # aniso-Gaussian + fock-reg 20K run at gamma*=0.30
DELTA_MIN_PPL           = 1.0    # decision threshold (see notebook header)

# ── V_theta: Anisotropic Gaussian (diagonal + low-rank precision) ──
V_THETA_VARIANT             = 'aniso_gaussian'
V_THETA_N_HEADS             = 4        # one bank per xi channel
V_THETA_WELLS_PER_HEAD      = 8        # Gaussian wells per bank
V_THETA_DEPTH_CONDITION     = True
V_THETA_DEPTH_CODE_INIT_STD = 0.02
ANISO_RANK                  = 4        # low-rank factor r for B_k in R^{d x r}

# ── Xi channels ──
XI_CHANNELS    = 4
XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]
XI_LEARNABLE   = True

# ── PARFLM ──
V_PHI_KIND     = 'structural_competitive'
TOP_K          = 8

# ── Training (matched to the second-order anchor) ──
STEPS          = 20_000
BATCH          = 4
GRAD_ACCUM     = 4    # effective batch = 16
BLOCK          = 512
LR             = 5e-4
WD             = 0.01
WARMUP         = 400
GRAD_CLIP      = 1.0
FOCK_GRAD_CLIP = 0.5
LAMBDA_V       = 1e-2

# ── Fock coupling regularisation ──
LAMBDA_FOCK_REG = 5e-3    # log-barrier strength for alpha engagement
FOCK_REG_EPS    = 1e-6    # epsilon inside log(alpha_k + eps)

# ── Evaluation ──
EVAL_INTERVAL  = 400
EVAL_ITERS     = 40
LOG_INTERVAL   = 50

# ── Checkpointing ──
CHECKPOINT_INTERVAL = 1000

# ── Causal probes ──
CAUSAL_PROBE_INTERVAL       = 4000
TRAINED_LEAK_PROBE_INTERVAL = 8000
TRAINED_LEAK_PROBE_K        = 256
TRAINED_LEAK_PROBE_PAIRS    = 2

total_wells = V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD
aniso_params_per_well = D * ANISO_RANK
print(f'Fock-G1 FIRST-ORDER ablation — Aniso-Gaussian on TinyStories d={D} L={L}')
print(f'  first_order={FIRST_ORDER}  gamma*={FIXED_GAMMA} (fixed; defines beta, no inertia)')
print(f'  2nd-order anchor PPL={SECOND_ORDER_ANCHOR_PPL}  Delta_min={DELTA_MIN_PPL}')
print(f'  V_theta: {V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells = '
      f'{total_wells} total attractors  (aniso rank r={ANISO_RANK})')
print(f'  Fock coupling reg: lambda={LAMBDA_FOCK_REG}  eps={FOCK_REG_EPS}')
print(f'  xi_channels={XI_CHANNELS}  alpha_inits={XI_ALPHA_INITS}')
print(f'  steps={STEPS}  batch={BATCH}  block={BLOCK}  lr={LR}  seed={SEED}')


In [ ]:
# ── Cell 1: Environment + Drive Mount ──────────────────────────────
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _sh('pip install -q transformers huggingface_hub pyarrow matplotlib')

    # NEW GDrive location: separate from isotropic experiment
    GDRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_fock_g1_aniso_gaussian_fockreg_tinystories')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)

    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)

    RESULTS_DIR = GDRIVE_ROOT / 'results'
    RESULTS_DIR.mkdir(exist_ok=True)
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    RESULTS_DIR = (REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup'
                   / 'results' / 'fock_g1_aniso_gaussian_fockreg_tinystories')
    for d in [DATA_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

RUN_DIR = RESULTS_DIR / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f'DATA_DIR    = {DATA_DIR}')
print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'RUN_DIR     = {RUN_DIR}')


In [ ]:
# ── Cell 2: GPU Check + Imports ────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print('WARNING: No GPU detected. This notebook requires CUDA.')

from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse

print('Model imports OK')


In [ ]:
# ── Cell 3: Data Loading (TinyStories) ─────────────────────────────
from data_module import get_batch, load_tiny_stories

train_ids, val_ids = load_tiny_stories(max_train_tokens=5_000_000)
print(f'train: {len(train_ids):,} tokens   val: {len(val_ids):,} tokens')

rng = np.random.default_rng(SEED)


In [ ]:
# ── Cell 4: Anisotropic Gaussian V_theta ───────────────────────────
#
# Extends the base MixtureGaussianVTheta with low-rank cross-correlations.
#
# Precision per well:  Sigma_k^{-1} = diag(a_k) + B_k @ B_k^T
# where B_k in R^{d x r} is a learned low-rank factor.
#
# The potential and its gradient remain closed-form:
#   V_k = -w_k exp(-0.5 [(a_k*diff^2).sum() + ||B_k^T diff||^2])
#   grad_h V_k = w_k [a_k*diff + B_k(B_k^T diff)] exp(exponent)
#
# Forces remain bounded (same diff*exp(-r^2/2) decay).


class AnisotropicMixtureGaussianVTheta(nn.Module):

    def __init__(self, d: int, K: int = 8, rank: int = 4,
                 w_scale: float = 1.0, xi_d=None,
                 init_log_precision=None, precision_max=None,
                 force_norm_max=None):
        super().__init__()
        self.d = d
        self.K = K
        self.rank = rank
        self.w_scale = w_scale
        self._precision_max = precision_max
        self._force_norm_max = force_norm_max
        in_d = xi_d if xi_d is not None else d

        self.mu_proj = nn.Linear(in_d, K * d)
        self.a_proj = nn.Linear(in_d, K * d)
        self.w_proj = nn.Linear(in_d, K)
        self.B_proj = nn.Linear(in_d, K * d * rank)

        self._init_weights(init_log_precision)

    def _init_weights(self, init_log_precision):
        nn.init.xavier_uniform_(self.mu_proj.weight)
        nn.init.zeros_(self.mu_proj.bias)
        nn.init.zeros_(self.a_proj.weight)
        if init_log_precision is not None:
            self.a_proj.bias.data.fill_(init_log_precision)
        else:
            self.a_proj.bias.data.fill_(0.0)
        nn.init.zeros_(self.w_proj.weight)
        nn.init.zeros_(self.w_proj.bias)
        nn.init.normal_(self.B_proj.weight, std=0.01)
        nn.init.zeros_(self.B_proj.bias)

    def _components(self, xi):
        lead = xi.shape[:-1]
        mu = self.mu_proj(xi).view(*lead, self.K, self.d)
        a = (F.softplus(self.a_proj(xi)) + 1e-4).view(*lead, self.K, self.d)
        if self._precision_max is not None:
            a = a.clamp(max=self._precision_max)
        w = F.softmax(self.w_proj(xi), dim=-1) * self.w_scale
        B = self.B_proj(xi).view(*lead, self.K, self.d, self.rank)
        return mu, a, w, B

    def forward(self, xi, h):
        mu, a, w, B = self._components(xi)
        h_e = h.unsqueeze(-2)
        diff = h_e - mu

        diag_term = (a * diff * diff).sum(dim=-1)
        Bt_diff = torch.einsum('...kd,...kdr->...kr', diff, B)
        lr_term = (Bt_diff * Bt_diff).sum(dim=-1)

        exponent = -0.5 * (diag_term + lr_term)
        bumps = w * torch.exp(exponent)
        return -bumps.sum(dim=-1, keepdim=True)

    def analytical_grad(self, xi, h):
        mu, a, w, B = self._components(xi)
        h_e = h.unsqueeze(-2)
        diff = h_e - mu

        diag_term = (a * diff * diff).sum(dim=-1)
        Bt_diff = torch.einsum('...kd,...kdr->...kr', diff, B)
        lr_term = (Bt_diff * Bt_diff).sum(dim=-1)

        exponent = -0.5 * (diag_term + lr_term)
        g = w * torch.exp(exponent)

        grad_diag = a * diff
        grad_lr = torch.einsum('...kdr,...kr->...kd', B, Bt_diff)
        per_comp = (grad_diag + grad_lr) * g.unsqueeze(-1)

        if self._force_norm_max is not None:
            norms = per_comp.norm(dim=-1, keepdim=True).clamp(min=1e-8)
            scale = (self._force_norm_max / norms).clamp(max=1.0)
            per_comp = per_comp * scale

        return per_comp.sum(dim=-2)

    def attractor_centres(self, xi):
        lead = xi.shape[:-1]
        return self.mu_proj(xi).view(*lead, self.K, self.d)


class AnisotropicMultiContextGaussianVTheta(nn.Module):

    def __init__(self, d, K, n_ctx, rank=4, w_scale=1.0,
                 init_log_precision=None, precision_max=None,
                 force_norm_max=None):
        super().__init__()
        self.d = d
        self.K = K
        self.n_ctx = n_ctx
        self.banks = nn.ModuleList(
            AnisotropicMixtureGaussianVTheta(
                d=d, K=K, rank=rank, w_scale=w_scale, xi_d=d,
                init_log_precision=init_log_precision,
                precision_max=precision_max,
                force_norm_max=force_norm_max,
            )
            for _ in range(n_ctx)
        )

    def forward(self, xis, h):
        out = self.banks[0](xis[..., 0, :], h)
        for m in range(1, self.n_ctx):
            out = out + self.banks[m](xis[..., m, :], h)
        return out

    def analytical_grad(self, xis, h):
        out = self.banks[0].analytical_grad(xis[..., 0, :], h)
        for m in range(1, self.n_ctx):
            out = out + self.banks[m].analytical_grad(xis[..., m, :], h)
        return out

    def attractor_centres(self, xis):
        cs = [self.banks[m].attractor_centres(xis[..., m, :])
              for m in range(self.n_ctx)]
        return torch.cat(cs, dim=-2)


class AnisotropicDepthConditionedGaussianVTheta(nn.Module):

    def __init__(self, d, K, n_ctx, n_layers, rank=4,
                 w_scale=1.0, init_log_precision=None,
                 precision_max=None, force_norm_max=None,
                 code_init_std=0.02):
        super().__init__()
        self.d = d
        self.K = K
        self.n_ctx = n_ctx
        self.n_layers = n_layers
        self.bank = AnisotropicMultiContextGaussianVTheta(
            d=d, K=K, n_ctx=n_ctx, rank=rank, w_scale=w_scale,
            init_log_precision=init_log_precision,
            precision_max=precision_max,
            force_norm_max=force_norm_max,
        )
        self.depth_code = nn.Parameter(
            torch.randn(n_layers, n_ctx, d) * code_init_std
        )
        self._active_layer: int = 0

    @property
    def banks(self):
        return self.bank.banks

    def set_active_layer(self, layer_idx: int):
        self._active_layer = int(layer_idx)

    def _shift(self, xis):
        g = self._active_layer
        if not (0 <= g < self.n_layers):
            g = g % self.n_layers
        code = self.depth_code[g]
        lead = xis.dim() - 2
        code = code.view(*([1] * lead), self.n_ctx, self.d)
        return xis + code

    def forward(self, xis, h):
        return self.bank(self._shift(xis), h)

    def analytical_grad(self, xis, h):
        return self.bank.analytical_grad(self._shift(xis), h)

    def attractor_centres(self, xis):
        return self.bank.attractor_centres(self._shift(xis))


def install_aniso_depth_routing(model):
    vt = model.V_theta
    if not hasattr(vt, 'set_active_layer'):
        return

    _orig_layer_step = model._layer_step.__func__

    def _patched_layer_step(self, h, h_prev, m_b, gamma, layer_idx=0,
                            x_tokens=None, **kw):
        if hasattr(self.V_theta, 'set_active_layer'):
            self.V_theta.set_active_layer(layer_idx)
        return _orig_layer_step(self, h, h_prev, m_b, gamma,
                                layer_idx=layer_idx,
                                x_tokens=x_tokens, **kw)

    import types
    model._layer_step = types.MethodType(_patched_layer_step, model)


# Smoke test
_test_xi = torch.randn(2, 4, 4, D)
_test_h = torch.randn(2, 4, D)
_test_bank = AnisotropicDepthConditionedGaussianVTheta(
    d=D, K=V_THETA_WELLS_PER_HEAD, n_ctx=V_THETA_N_HEADS,
    n_layers=L, rank=ANISO_RANK,
)
_test_V = _test_bank(_test_xi, _test_h)
_test_G = _test_bank.analytical_grad(_test_xi, _test_h)
print(f'Smoke test:  V shape={_test_V.shape}  grad shape={_test_G.shape}')
print(f'  V range: [{_test_V.min().item():.4f}, {_test_V.max().item():.4f}]')
print(f'  grad norm: {_test_G.norm(dim=-1).mean().item():.4f}')
n_aniso = sum(p.numel() for p in _test_bank.parameters())
print(f'  Aniso V_theta params: {n_aniso:,}')
del _test_xi, _test_h, _test_bank, _test_V, _test_G
print('Anisotropic Gaussian V_theta OK')


In [ ]:
# ── Cell 5b: Fock-G1 First-Order Ablation Model ────────────────────
#
# The ONLY change vs. the second-order FockMultiXiPARFLM is that the per-layer
# velocity memory delta = h - h_prev is forced to zero, collapsing the damped
# velocity-Verlet update to a pure gradient step:
#
#   second order:  h_new = LN( h + delta/(1+dt*gamma) + dt^2/(m_b*(1+dt*gamma)) * f )
#   Fock-G1:       h_new = LN( h + beta * f ),  beta = dt^2/(m_b*(1+dt*gamma*))
#
# Achieved by overriding _fock_layer_step so it passes h_prev := h down to the
# inherited token dynamics (super()._fock_layer_step), making delta identically
# zero at every layer. Every other channel -- V_theta, V_phi, xi, register
# creation/destruction, the reverse channel, fock-reg -- is inherited unchanged,
# so the ablation isolates the inertial term only.


class FockG1MultiXiPARFLM(FockMultiXiPARFLM):
    """First-order (gradient-flow) ablation of Fock v2.1 aniso-Gaussian.

    See companion_notes/Implicit_vs_Explicit_Damping_and_the_First_vs_Second_
    Order_Dynamics_Hypothesis.md section 6 (protocol "Fock-G1").
    """

    def _fock_layer_step(self, h, h_prev, r, salience, m_b, gamma, dt, layer_idx):
        # First order: discard the incoming velocity memory (h_prev := h) so
        # that delta = h - h_prev = 0 in the inherited damped-Verlet update.
        return super()._fock_layer_step(
            h, h, r, salience, m_b, gamma, dt, layer_idx,
        )


# ── Smoke test: first-order and second-order differ; FO output is finite ──
def _fock_g1_smoke():
    _cfg = FockMultiXiPARFConfig(
        vocab_size=257, d=32, max_len=64, L=4, v_hidden=64, v_depth=2, dt=1.0,
        mass_mode='global', fixed_gamma=0.30, causal_force=True,
        ln_after_step=True,
        xi_channels=3, xi_alpha_inits=[0.5, 0.9, 0.99], xi_learnable=True,
        xi_alpha_init_mode='explicit',
        v_phi_kind='structural_competitive', top_k=4, score_head_hidden=8,
        gumbel_noise=False,
        use_gathered_v_phi=True, use_layer_checkpoint=False,
        fock_version='v2', n_registers=8,
        register_salience_decay=0.5, register_salience_threshold=0.005,
        creation_gate_hidden=16, stack_discipline=True, d_k=16,
        tau_create_init=8.0,
        reverse_channel=True, per_register_tau=True, per_register_keys=True,
        ortho_register_init=True, prefix_causal_registers=True,
    )
    torch.manual_seed(0)
    m2 = FockMultiXiPARFLM(_cfg)          # second order
    torch.manual_seed(0)
    m1 = FockG1MultiXiPARFLM(_cfg)        # first order
    m1.load_state_dict(m2.state_dict())   # identical weights

    x = torch.randint(0, 257, (2, 16))
    y = torch.randint(0, 257, (2, 16))

    m1.eval(); m2.eval()
    # The conservative force uses autograd.grad internally, so grad must be
    # enabled even for an eval-mode forward (mirrors evaluate_model()).
    with torch.enable_grad():
        l2, _ = m2(x, y)
        l1, _ = m1(x, y)
    l1 = l1.detach(); l2 = l2.detach()
    max_diff = (l1 - l2).abs().max().item()
    print(f'[Fock-G1 smoke] max|logit_1st - logit_2nd| = {max_diff:.4e} '
          f'(>0: dynamics diverge from layer 1 onward)')
    print(f'[Fock-G1 smoke] first-order logits finite: '
          f'{bool(torch.isfinite(l1).all())}')

    # gradient flow through the first-order model
    m1.train()
    _, loss1 = m1(x, y)
    loss1.backward()
    _n_grad = sum(1 for p in m1.parameters()
                  if p.grad is not None and p.grad.abs().sum() > 0)
    print(f'[Fock-G1 smoke] loss={loss1.item():.4f}  '
          f'params_with_grad={_n_grad}')

    assert bool(torch.isfinite(l1).all()), 'first-order logits not finite'
    assert max_diff > 0, 'first- and second-order identical (delta not zeroed?)'


_fock_g1_smoke()
print('Fock-G1 first-order model OK')


In [ ]:
# ── Cell 5: Model Builder + Anisotropic Gaussian V_theta ───────────

# ── Logfreq surprisal ──
LOGFREQ_PATH = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_tinystories.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_tinystories.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, surprisal)
    print(f'Built logfreq from train_ids; saved to {LOGFREQ_FILE}')

print(f'Logfreq: {LOGFREQ_FILE}')

# ── Build model ──
torch.manual_seed(SEED)

cfg = FockMultiXiPARFConfig(
    vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
    L=L, v_hidden=1024, v_depth=3, dt=DT,
    mass_mode='logfreq',
    logfreq_path=str(LOGFREQ_FILE),
    logfreq_init_alpha=0.1,
    init_gamma=1.0,
    fixed_gamma=FIXED_GAMMA,
    causal_force=True,
    ln_after_step=True,
    xi_channels=XI_CHANNELS,
    xi_alpha_inits=XI_ALPHA_INITS,
    xi_learnable=XI_LEARNABLE,
    xi_alpha_init_mode='explicit',
    v_phi_kind=V_PHI_KIND,
    v_phi_phi_hidden=128,
    v_phi_theta_hidden=128,
    top_k=TOP_K,
    score_head_hidden=32,
    gumbel_tau_init=1.0,
    gumbel_tau_min=0.3,
    gumbel_noise=True,
    use_gathered_v_phi=True,
    use_layer_checkpoint=True,
    ln_before_distance=True,
    per_layer_v_phi_scale=True,
    fock_version='v2',
    n_registers=16,
    register_salience_decay=0.5,
    register_salience_threshold=0.005,
    creation_gate_hidden=64,
    stack_discipline=True,
    d_k=64,
    tau_create_init=8.0,
    reverse_channel=True,
    per_register_tau=True,
    per_register_keys=True,
    ortho_register_init=True,
    prefix_causal_registers=True,
)
model = FockG1MultiXiPARFLM(cfg).to(DEVICE)

n_total_before = sum(p.numel() for p in model.parameters())
n_v_theta_before = sum(p.numel() for p in model.V_theta.parameters())
print(f'Before swap: total={n_total_before:,}  V_theta(MLP)={n_v_theta_before:,}')

# ── Swap in Anisotropic Gaussian V_theta ──
import math as _m
model.V_theta = AnisotropicDepthConditionedGaussianVTheta(
    d=D,
    K=V_THETA_WELLS_PER_HEAD,
    n_ctx=V_THETA_N_HEADS,
    n_layers=L,
    rank=ANISO_RANK,
    w_scale=1.0,
    init_log_precision=-_m.log(D),
    precision_max=2.0 / D,
    code_init_std=V_THETA_DEPTH_CODE_INIT_STD,
).to(DEVICE)
install_aniso_depth_routing(model)

n_total = sum(p.numel() for p in model.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
print(f'After swap:')
print(f'  total params   = {n_total:,}')
print(f'  V_theta params = {n_v_theta:,}  ({n_v_theta/n_total*100:.1f}%)')
print(f'  V_theta: {V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells '
      f'= {V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} attractors')
print(f'  Aniso rank: {ANISO_RANK}  depth-conditioned: {V_THETA_DEPTH_CONDITION}')
print(f'  xi_alpha init: {model.xi_alpha_values()}')
assert model._gamma_value is not None, (
    'Fock-G1 requires fixed gamma (fixed_gamma set) so beta is a constant')
_gamma_star = float(model.gamma.item())
print(f'First-order (Fock-G1): velocity memory zeroed (delta := 0)')
print(f'  gamma*={_gamma_star:.3f} (fixed)  =>  '
      f'beta = dt^2/(m_b*(1+dt*gamma*)) = {DT*DT:.3f}/(m_b*{1.0+DT*_gamma_star:.3f})'
      f'  [per-token m_b]')
print(f'Model builder OK')


In [ ]:
# ── Cell 6: Training + Evaluation Helpers ──────────────────────────

EVAL_MICRO_BATCH = 4


def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def fock_coupling_reg(model, lam, eps):
    """Log-barrier regularisation on alpha_k coupling strengths.

    L_fock = -lam * sum_k log(alpha_k + eps)

    Strong gradient when alpha_k is small, negligible at high coupling.
    """
    alphas = model.xi_module.alpha  # sigmoid(raw_alpha), differentiable
    return -lam * torch.log(alphas + eps).sum()


def forward_with_vreg(model, x, targets, lambda_v, lambda_fock, fock_eps):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = model.compute_logits(h_L)
    loss_ntp = F.cross_entropy(
        logits.float().reshape(-1, cfg.vocab_size),
        targets.reshape(-1),
    )
    v_reg_value = torch.tensor(0.0, device=x.device)
    fock_reg_value = torch.tensor(0.0, device=x.device)

    loss = loss_ntp

    if lambda_v > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        v_reg_value = (V_vals.float() ** 2).mean()
        loss = loss + lambda_v * v_reg_value

    if lambda_fock > 0:
        fock_reg_value = fock_coupling_reg(model, lambda_fock, fock_eps)
        loss = loss + fock_reg_value

    return logits, loss, loss_ntp, v_reg_value, fock_reg_value


@torch.no_grad()
def evaluate_model():
    model.eval()
    micro = min(EVAL_MICRO_BATCH, BATCH)
    n_micro = max(1, BATCH // micro)
    losses = []
    for _ in range(EVAL_ITERS):
        micro_losses = []
        for _m in range(n_micro):
            xb, yb = get_batch(val_ids, micro, BLOCK, rng)
            x = torch.from_numpy(xb).to(DEVICE)
            y = torch.from_numpy(yb).to(DEVICE)
            with torch.enable_grad():
                _, loss = model(x, y)
            micro_losses.append(loss.item())
            del loss, x, y
        losses.append(float(np.mean(micro_losses)))
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
    model.train()
    return float(np.mean(losses))


print('Training helpers OK')


In [ ]:
# ── Cell 7: Two-Stage Causal Probes ────────────────────────────────

def run_causal_probe(step_num):
    """Stage 1: lightweight architectural causal probe (CPU, float64)."""
    import math as _math
    from model_gaussian_vtheta import (
        DepthConditionedMultiContextGaussianVTheta,
        install_depth_routing,
    )
    _PROBE_VOCAB, _PROBE_D, _PROBE_L = 101, 32, 4
    _PROBE_T, _PROBE_M, _PROBE_XI = 48, 8, 3
    _PROBE_WELLS = 4

    _logfreq_probe = Path('/tmp/causal_probe_logfreq.npy')
    np.save(_logfreq_probe, np.full(_PROBE_VOCAB, 5.0, dtype=np.float32))

    _probe_cfg = FockMultiXiPARFConfig(
        vocab_size=_PROBE_VOCAB, d=_PROBE_D, max_len=64, L=_PROBE_L,
        v_hidden=64, v_depth=1, dt=0.1,
        mass_mode='global', causal_force=True,
        ln_after_step=True,
        xi_channels=_PROBE_XI, xi_alpha_inits=[0.5, 0.9, 0.99],
        xi_learnable=False, xi_alpha_init_mode='explicit',
        fock_version='v2', n_registers=_PROBE_M,
        register_salience_decay=0.5,
        register_salience_threshold=0.005,
        creation_gate_hidden=16, stack_discipline=True,
        d_k=16, tau_create_init=8.0,
        reverse_channel=True,
        per_register_tau=True, per_register_keys=True,
        ortho_register_init=True,
        prefix_causal_registers=True,
    )
    torch.manual_seed(1234)
    _probe_model = FockG1MultiXiPARFLM(_probe_cfg)

    _probe_model.V_theta = DepthConditionedMultiContextGaussianVTheta(
        d=_PROBE_D, K=_PROBE_WELLS, n_ctx=_PROBE_XI, n_layers=_PROBE_L,
        w_scale=1.0, init_log_precision=-_math.log(_PROBE_D),
        precision_max=2.0 / _PROBE_D, code_init_std=0.02,
    )
    install_depth_routing(_probe_model)
    _probe_model.double().eval()

    with torch.no_grad():
        for n, p in _probe_model.named_parameters():
            if 'reverse_channel_scale' in n:
                p.fill_(5.0)

    _t_p = _PROBE_T // 2
    _prng = np.random.default_rng(7)
    _x1 = torch.from_numpy(_prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T))).long()
    _x2 = _x1.clone()
    _x2[:, _t_p:] = torch.from_numpy(
        _prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T - _t_p))).long()

    _max_delta = 0.0
    for mode_name, use_train in [('eval', False), ('train', True)]:
        if use_train:
            _probe_model.train()
            torch.manual_seed(99)
        else:
            _probe_model.eval()
        with torch.enable_grad():
            _la = _probe_model(_x1)[0].detach()
        if use_train:
            torch.manual_seed(99)
        with torch.enable_grad():
            _lb = _probe_model(_x2)[0].detach()
        delta = float((_la[:, :_t_p] - _lb[:, :_t_p]).abs().max().item())
        _max_delta = max(_max_delta, delta)

    _passed = (_max_delta == 0.0)
    del _probe_model, _la, _lb, _x1, _x2
    gc.collect()

    status = 'PASS' if _passed else '*** FAIL ***'
    print(f'\n[causal probe] step {step_num:,}  max|dlogit|={_max_delta:.3e}  [{status}]')
    if not _passed:
        print('[causal probe] WARNING: nonzero future sensitivity detected!')
    return _passed, _max_delta


def run_trained_leak_probe(step_num):
    """Stage 2: trained-scale leak probe + honest PPL on the live model."""
    _debug_dir = str(CA_DIR / 'scaleup' / 'debug')
    if _debug_dir not in sys.path:
        sys.path.insert(0, _debug_dir)
    from fock_trained_leak_probe import probe_trained_leak, honest_ppl_test

    print(f'\n{"="*64}')
    print(f'[trained leak probe] step {step_num:,} — running on live model')
    print(f'{"="*64}')

    probe_res = probe_trained_leak(
        model, val_ids, device=DEVICE, context=BLOCK,
        n_pairs=TRAINED_LEAK_PROBE_PAIRS, use_float64=False)

    honest_res = honest_ppl_test(
        model, val_ids, k=TRAINED_LEAK_PROBE_K,
        context=BLOCK, batch=BATCH, device=DEVICE)

    model.train()

    result = {
        'step': step_num,
        'event': 'trained_leak_probe',
        'probe_max_dlogit_past': probe_res['max_dlogit_past'],
        'probe_mean_dnll_past_nats': round(probe_res['mean_dnll_past'], 6),
        'probe_gate_zero_control': probe_res['gate_zero_control'],
        'honest_k': honest_res['k'],
        'ppl_mid_window_standard': round(honest_res['ppl_mid_window'], 4),
        'ppl_last_pos_leak_free': round(honest_res['ppl_last_pos'], 4),
        'paired_diff_nats': round(honest_res['paired_diff_nats'], 6),
        'paired_diff_se': round(honest_res['paired_diff_se'], 6),
    }

    _leak_status = 'CLEAN' if result['paired_diff_nats'] < 0.1 else 'LEAK DETECTED'
    print(f'\n[trained leak probe] step {step_num:,}  '
          f'honest_PPL={result["ppl_last_pos_leak_free"]:.2f}  '
          f'standard_PPL={result["ppl_mid_window_standard"]:.2f}  '
          f'diff={result["paired_diff_nats"]:+.4f} nats  [{_leak_status}]')
    return result


print('Two-stage causal probes OK')


In [ ]:
# ── Cell 8: Training Loop ──────────────────────────────────────────

resume_step = 0
_latest_ckpt_path = RUN_DIR / 'ckpt_latest.pt'
_best_ckpt_path = RUN_DIR / 'ckpt_best.pt'

opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, betas=(0.9, 0.95), weight_decay=WD,
)

if _latest_ckpt_path.exists():
    ckpt = torch.load(_latest_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    if 'optimizer_state_dict' in ckpt:
        try:
            opt.load_state_dict(ckpt['optimizer_state_dict'])
            print(f'Optimizer state restored.')
        except (ValueError, KeyError) as e:
            print(f'[info] Optimizer state incompatible: {e}')
    resume_step = ckpt.get('step', 0)
    print(f'Resumed from step {resume_step:,}  '
          f'(PPL {ckpt.get("val_ppl", "?")})  [{_latest_ckpt_path.name}]')
    del ckpt

best_val_ppl = float('inf')
if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored best PPL from previous session: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] Could not read best checkpoint: {e}')

if resume_step > 0:
    for _ in range(resume_step * GRAD_ACCUM):
        get_batch(train_ids, BATCH, BLOCK, rng)

log_path = RUN_DIR / 'training_log.jsonl'
log_f = log_path.open('a')

SPIKE_THRESHOLD  = 500.0
SPIKE_COOLDOWN   = 20
_last_spike_step = -10**9

val_ppl = best_val_ppl


def _log_write(record):
    log_f.write(json.dumps(record) + '\n')
    log_f.flush()


model.train()
t0 = time.time()
t_session = time.time()

for step in range(resume_step, STEPS):
    for g in opt.param_groups:
        g['lr'] = lr_at(step)

    opt.zero_grad(set_to_none=True)
    step_loss_ntp = 0.0
    step_v_reg = 0.0
    step_fock_reg = 0.0
    step_loss_total = 0.0

    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        _, loss, loss_ntp, v_reg, fock_reg = forward_with_vreg(
            model, x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
        (loss / GRAD_ACCUM).backward()
        step_loss_ntp   += loss_ntp.item() / GRAD_ACCUM
        step_v_reg      += v_reg.item()    / GRAD_ACCUM
        step_fock_reg   += fock_reg.item()  / GRAD_ACCUM
        step_loss_total += loss.item()      / GRAD_ACCUM

    # ── Per-group gradient clipping ──
    _groups = {}
    for _n, _p in model.named_parameters():
        if _p.grad is None:
            continue
        if _n.startswith('fock_layers.'):
            _g = 'fock'
        elif _n.startswith('V_phi.'):
            _g = 'vphi'
        elif _n.startswith('V_theta.'):
            _g = 'vtheta'
        elif 'reverse_channel_scale' in _n:
            _g = 'rc_scale'
        else:
            _g = 'other'
        _groups.setdefault(_g, []).append(_p)

    _CLIP_LIMITS = {
        'fock': FOCK_GRAD_CLIP,
        'vphi': GRAD_CLIP * 0.3,
        'vtheta': GRAD_CLIP,
        'rc_scale': GRAD_CLIP,
        'other': GRAD_CLIP,
    }
    _group_norms = {}
    for _g, _params in _groups.items():
        _group_norms[_g] = float(
            nn.utils.clip_grad_norm_(_params, _CLIP_LIMITS.get(_g, GRAD_CLIP)))

    _all_params = [p for p in model.parameters() if p.grad is not None]
    grad_norm = sum(p.grad.data.norm().item() ** 2 for p in _all_params) ** 0.5

    _filtered = {k: v for k, v in _group_norms.items() if k != 'rc_scale'}
    _top_group = max(_filtered, key=_filtered.get) if _filtered else 'none'
    _top_norm = _filtered.get(_top_group, 0.0)

    opt.step()

    for bank in model.V_theta.banks:
        if hasattr(bank, 'clamp_params'):
            bank.clamp_params()

    # Spike detection
    if (grad_norm > SPIKE_THRESHOLD
            and (step - _last_spike_step) >= SPIKE_COOLDOWN):
        _last_spike_step = step
        print(f'\n[spike] step {step+1}: pre-clip grad={grad_norm:.1f}  '
              f'top[{_top_group}]={_top_norm:.1f}  '
              f'ntp={step_loss_ntp:.4f}  v_reg={step_v_reg:.4f}  '
              f'fock_reg={step_fock_reg:.4f}')
        _log_write({
            'step': step + 1, 'event': 'grad_spike',
            'pre_clip_grad_norm': round(grad_norm, 2),
            'ntp': round(step_loss_ntp, 4),
            'v_reg': round(step_v_reg, 4),
            'fock_reg': round(step_fock_reg, 4),
        })

    # Causal probes
    if CAUSAL_PROBE_INTERVAL > 0 and (step + 1) % CAUSAL_PROBE_INTERVAL == 0:
        _cp_passed, _cp_delta = run_causal_probe(step + 1)
        _log_write({
            'step': step + 1, 'event': 'causal_probe',
            'causal_probe_passed': _cp_passed,
            'causal_probe_max_delta': _cp_delta,
        })

    if TRAINED_LEAK_PROBE_INTERVAL > 0 and (step + 1) % TRAINED_LEAK_PROBE_INTERVAL == 0:
        _tlp = run_trained_leak_probe(step + 1)
        _log_write(_tlp)

    # Logging
    if (step + 1) % LOG_INTERVAL == 0 or step == resume_step:
        elapsed = time.time() - t_session
        steps_done = step + 1 - resume_step
        sec_per_step = elapsed / max(steps_done, 1)
        remaining = (STEPS - step - 1) * sec_per_step
        alphas_str = ','.join(f'{a:.3f}' for a in model.xi_alpha_values())
        print(f'step {step+1:>5}/{STEPS}  '
              f'ntp={step_loss_ntp:.4f}  v_reg={step_v_reg:.4f}  '
              f'fock_reg={step_fock_reg:.4f}  '
              f'lr={lr_at(step):.2e}  grad={grad_norm:.3f}  '
              f'top[{_top_group}]={_top_norm:.1f}  '
              f'gamma={model.gamma.item():.3f}  '
              f'alpha=[{alphas_str}]  '
              f'{elapsed:.0f}s (~{remaining/60:.1f}m remaining)')
        _log_write({
            'step': step + 1, 'train_loss': step_loss_ntp,
            'v_reg': step_v_reg, 'fock_reg': step_fock_reg,
            'total_loss': step_loss_total,
            'lr': lr_at(step), 'grad_norm': grad_norm,
            'gamma': model.gamma.item(),
            'xi_alphas': model.xi_alpha_values(),
        })

    # Evaluation
    if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == STEPS:
        val_loss = evaluate_model()
        val_ppl = math.exp(val_loss)
        is_best = val_ppl < best_val_ppl
        if is_best:
            best_val_ppl = val_ppl
        best_marker = '  *** NEW BEST ***' if is_best else ''
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}  '
              f'best={best_val_ppl:.2f}{best_marker}')
        _log_write({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_val_ppl,
        })
        if is_best:
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': opt.state_dict(),
                'step': step + 1, 'val_loss': val_loss,
                'val_ppl': val_ppl, 'gamma': model.gamma.item(),
                'xi_alphas': model.xi_alpha_values(),
                'v_theta_variant': V_THETA_VARIANT, 'first_order': True, 'gamma_star': FIXED_GAMMA,
                'aniso_rank': ANISO_RANK,
                'lambda_fock_reg': LAMBDA_FOCK_REG,
            }, str(_best_ckpt_path))

    # Periodic checkpoint
    if (step + 1) % CHECKPOINT_INTERVAL == 0 or (step + 1) == STEPS:
        _ckpt_ppl = val_ppl if (step + 1) % EVAL_INTERVAL == 0 else best_val_ppl
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'step': step + 1,
            'val_ppl': _ckpt_ppl,
            'best_val_ppl': best_val_ppl,
            'gamma': model.gamma.item(),
            'xi_alphas': model.xi_alpha_values(),
            'v_theta_variant': V_THETA_VARIANT, 'first_order': True, 'gamma_star': FIXED_GAMMA,
            'aniso_rank': ANISO_RANK,
            'lambda_fock_reg': LAMBDA_FOCK_REG,
        }, str(_latest_ckpt_path))
        print(f'  [ckpt] saved {_latest_ckpt_path.name} at step {step+1}')

log_f.close()
print(f'\nTraining done.  total wall = {time.time()-t_session:.0f}s  '
      f'final_ppl = {val_ppl:.2f}  best_ppl = {best_val_ppl:.2f}')


In [ ]:
# ── Cell 9: Training Curve ─────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

eval_entries = []
alpha_entries = []
if log_path.exists():
    with open(log_path) as f:
        for line in f:
            try:
                e = json.loads(line)
                if 'val_ppl' in e and 'event' not in e:
                    eval_entries.append(e)
                if 'xi_alphas' in e and 'event' not in e:
                    alpha_entries.append(e)
            except Exception:
                pass

if eval_entries:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    steps_arr = [e['step'] for e in eval_entries]
    ppls = [e['val_ppl'] for e in eval_entries]
    ax.plot(steps_arr, ppls, 'o-',
            label=f'Fock-G1 (first-order) aniso-Gaussian r={ANISO_RANK}',
            linewidth=1.5, color='#C62828')
    ax.axhline(y=SECOND_ORDER_ANCHOR_PPL, color='green', linestyle='--',
               alpha=0.85,
               label=f'2nd-order anchor ({SECOND_ORDER_ANCHOR_PPL:.2f} PPL)')
    ax.axhline(y=9.70, color='blue', linestyle=':', alpha=0.5,
               label='MLP baseline (9.70 PPL)')
    ax.axhline(y=16.33, color='gray', linestyle=':', alpha=0.7,
               label='Isotropic Gaussian (16.33 PPL)')
    ax.set_xlabel('Step')
    ax.set_ylabel('Val PPL')
    ax.set_title(f'Fock-G1 first-order ablation — TinyStories d={D}')
    ax.legend()
    ax.grid(True, alpha=0.3)

    if alpha_entries:
        ax = axes[1]
        a_steps = [e['step'] for e in alpha_entries]
        n_ch = len(alpha_entries[0]['xi_alphas'])
        for k in range(n_ch):
            ax.plot(a_steps, [e['xi_alphas'][k] for e in alpha_entries],
                    'o-', label=f'alpha_{k+1}', markersize=2, linewidth=1.5)
        ax.set_xlabel('Step')
        ax.set_ylabel('alpha_k')
        ax.set_title('Fock coupling strengths (alpha_k)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim(-0.05, 1.05)

    plt.tight_layout()
    fig.savefig(RUN_DIR / 'training_curve_fock_g1.png', dpi=150)
    plt.show()
    print(f'Saved: {RUN_DIR / "training_curve_fock_g1.png"}')
else:
    print('No eval data to plot.')


In [ ]:
# ── Cell 10: V_theta Landscape Diagnostics ─────────────────────────
v_samples = []
model.eval()
for _ in range(10):
    xb, _ = get_batch(val_ids, min(BATCH, 4), BLOCK, rng)
    x = torch.from_numpy(xb).to(DEVICE)
    with torch.enable_grad():
        h0 = model._embed(x)
        h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        v_samples.append(V_vals.detach().cpu().numpy().ravel())

v_all = np.concatenate(v_samples)
ls = {
    'mean': float(v_all.mean()),
    'std': float(v_all.std()),
    'min': float(v_all.min()),
    'max': float(v_all.max()),
    'range': float(v_all.max() - v_all.min()),
}
print(f'V_theta landscape stats:')
for k, v in ls.items():
    print(f'  {k:6s}: {v:.4f}')

ls_path = RUN_DIR / 'landscape_stats_aniso_gaussian.json'
with open(ls_path, 'w') as f:
    json.dump(ls, f, indent=2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(v_all, bins=80, edgecolor='none', alpha=0.8, color='#C62828')
ax.axvline(ls['mean'], color='blue', linestyle='--', alpha=0.6,
           label=f'mean={ls["mean"]:.2f}')
ax.set_xlabel('V_theta(xi, h)')
ax.set_ylabel('count')
ax.set_title(f'Aniso-Gaussian V_theta distribution (d={D}, r={ANISO_RANK})')
ax.legend()
plt.tight_layout()
fig.savefig(RUN_DIR / 'v_theta_hist_aniso_gaussian.png', dpi=150)
plt.show()
model.train()
print(f'Saved: {RUN_DIR / "v_theta_hist_aniso_gaussian.png"}')


In [ ]:
# ── Cell 11: Summary ───────────────────────────────────────────────
summary_path = RUN_DIR / 'summary_aniso_gaussian.md'
with open(summary_path, 'w') as f:
    f.write(f'# Fock-G1 First-Order Ablation (Aniso-Gaussian) — TinyStories\n\n')
    f.write(f'| Setting | Value |\n')
    f.write(f'|---------|-------|\n')
    f.write(f'| Dynamics order | First-order (Fock-G1, delta:=0) |\n')
    f.write(f'| gamma* (defines beta) | {FIXED_GAMMA} |\n')
    f.write(f'| V_theta variant | Anisotropic Gaussian (diag + low-rank r={ANISO_RANK}) |\n')
    f.write(f'| V_theta heads | {V_THETA_N_HEADS} |\n')
    f.write(f'| Wells per head | {V_THETA_WELLS_PER_HEAD} |\n')
    f.write(f'| Total attractors | {V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} |\n')
    f.write(f'| Anisotropic rank | {ANISO_RANK} |\n')
    f.write(f'| Depth conditioning | {V_THETA_DEPTH_CONDITION} |\n')
    f.write(f'| Fock coupling reg lambda | {LAMBDA_FOCK_REG} |\n')
    f.write(f'| xi_channels | {XI_CHANNELS} |\n')
    f.write(f'| lambda_V | {LAMBDA_V} |\n')
    f.write(f'| d | {D} |\n')
    f.write(f'| L | {L} |\n')
    f.write(f'| steps | {STEPS} |\n')
    f.write(f'| best PPL | {best_val_ppl:.2f} |\n')
    f.write(f'| V_theta params | {n_v_theta:,} |\n')
    f.write(f'| total params | {n_total:,} |\n')
    f.write(f'| xi_alpha_init | {XI_ALPHA_INITS} |\n')
    f.write(f'\n## Reference\n\n')
    f.write(f'| V_theta variant | Best PPL |\n')
    f.write(f'|---|---|\n')
    f.write(f'| MLP (ScalarPotentialMultiXi) | 9.70 |\n')
    f.write(f'| SQ3 Quadratic | 10.90 |\n')
    f.write(f'| Gaussian (isotropic, no fock-reg) | 16.33 |\n')
    f.write(f'| Aniso-Gaussian + fock-reg (2nd-order anchor, gamma*={FIXED_GAMMA}) | ~{SECOND_ORDER_ANCHOR_PPL:.2f} |\n')
    f.write(f'| **Fock-G1 first-order (this)** | **{best_val_ppl:.2f}** |\n')
    f.write(f'\n## Ablation decision (single seed — run 3 seeds for full protocol)\n\n')
    _delta = best_val_ppl - SECOND_ORDER_ANCHOR_PPL
    f.write(f'- Second-order anchor PPL: {SECOND_ORDER_ANCHOR_PPL:.2f}\n')
    f.write(f'- Fock-G1 first-order PPL: {best_val_ppl:.2f}\n')
    f.write(f'- Delta (FO - anchor): {_delta:+.2f} PPL  (Delta_min = {DELTA_MIN_PPL:.2f})\n')
    if _delta >= DELTA_MIN_PPL:
        _verdict = 'H1-consistent (inertia adds training-time value)'
    elif _delta <= -DELTA_MIN_PPL:
        _verdict = 'H-1-consistent (first-order wins)'
    else:
        _verdict = 'H0-consistent (within Delta_min; first-order matches)'
    f.write(f'- Single-seed reading: {_verdict}\n')

print(f'Summary saved to {summary_path}')
print(f'\nFinal result: Fock-G1 first-order = {best_val_ppl:.2f} PPL  '
      f'(2nd-order anchor = {SECOND_ORDER_ANCHOR_PPL:.2f}, '
      f'Delta = {best_val_ppl - SECOND_ORDER_ANCHOR_PPL:+.2f}, '
      f'Delta_min = {DELTA_MIN_PPL:.2f})')
